# MethylBERT pretraining: random sample masking inspection

Pulls a random line from the training dataset (the same one `fullrun.py` feeds the model) and prints the masking side-by-side with the original tokens, the labels, and the methylation track. Use this to sanity-check that:

1. The masking rate is roughly 15% (before k-mer neighbor expansion).
2. The 80/10/10 split between `[MASK]`, random, and keep-original looks right.
3. Special tokens (`<pad>`, `<sos>`, `<eos>`) are never masked.
4. The methylation track at masked positions has been set to `STATE_UNKNOWN` (the leak-prevention from `_masking`).
5. The label is `-100` everywhere except at masked positions.

In [1]:
import os, sys, glob, time, random
import numpy as np
import torch
from collections import Counter

print("Importing methylbert...", flush=True)
t0 = time.time()
from methylbert.data.vocab import MethylVocab
from methylbert.data.dataset import MethylBertPretrainDatasetBinary
from methylbert.data.nanopore.featurize import (
    STATE_UNMETH, STATE_METH, STATE_NON_CPG, STATE_UNKNOWN,
)
print(f"  imports took {time.time()-t0:.1f}s", flush=True)

#TRAIN_DIR = "/localstorage/bauerste/pretrain_data/pretrain_shards_4state_v1_balanced/train"
TRAIN_DIR = "/home/bauerste/pretrain_data/pretrain_shards_4state_v1_balanced/train"
SEQ_LEN, K = 510, 3

# Sanity check the directory BEFORE constructing the dataset.
n_json = len(glob.glob(os.path.join(TRAIN_DIR, "*.json")))
print(f"Found {n_json} shard JSON files in {TRAIN_DIR}", flush=True)
if n_json == 0:
    raise SystemExit("No shards found — wrong path?")

print("Building vocab...", flush=True)
t0 = time.time()
vocab = MethylVocab(k=K)
print(f"  vocab built in {time.time()-t0:.1f}s, size={len(vocab)}", flush=True)

print("Constructing dataset (memmapping shards)...", flush=True)
t0 = time.time()
dataset = MethylBertPretrainDatasetBinary(data_dir=TRAIN_DIR, vocab=vocab, seq_len=SEQ_LEN)
print(f"  dataset built in {time.time()-t0:.1f}s, rows={len(dataset):,}", flush=True)

STATE_NAMES = {STATE_UNMETH: "UNMETH", STATE_METH: "METH",
               STATE_NON_CPG: "NON_CPG", STATE_UNKNOWN: "UNKNOWN"}
print(f"Specials  pad={vocab.pad_index} unk={vocab.unk_index} "
      f"eos={vocab.eos_index} sos={vocab.sos_index} mask={vocab.mask_index}", flush=True)

Importing methylbert...
  imports took 0.0s
Found 168 shard JSON files in /home/bauerste/pretrain_data/pretrain_shards_4state_v1_balanced/train
Building vocab...
Building Vocab
  vocab built in 0.0s, size=69
Constructing dataset (memmapping shards)...
Loaded 168 shards, 122,999,997 total rows.
  dataset built in 0.3s, rows=122,999,997
Specials  pad=0 unk=1 eos=2 sos=3 mask=4


In [2]:
# Pick a random sample. Re-run this cell to see different ones.
idx = random.randrange(len(dataset))
sample = dataset[idx]

bert_input = sample["bert_input"].long()   # (seq_len+1,)  corrupted DNA
bert_label = sample["bert_label"].long()   # (seq_len+1,)  -100 except at MLM positions
bert_mask  = sample["bert_mask"].bool()    # (seq_len+1,)  True at MLM positions
methyl_seq = sample["methyl_seq"].long()   # (seq_len+1,)  methylation states

print(f"Picked global index: {idx}")
print(f"bert_input shape: {tuple(bert_input.shape)}")

Picked global index: 119386181
bert_input shape: (512,)


## Aggregate stats

How many positions are masked, what's the effective rate over non-special tokens, and how does the corruption split into `[MASK]` / random / keep-original?

In [ ]:
# Find the EOS position to bound 'real' content (everything after EOS is padding).
eos_positions = (bert_input == vocab.eos_index).nonzero(as_tuple=True)[0]
eos_pos = eos_positions[0].item() if len(eos_positions) > 0 else len(bert_input) - 1

# Non-special content positions (excludes <sos> at 0 and everything from <eos> onward).
content_slice = slice(1, eos_pos)
n_content    = eos_pos - 1
n_corrupted  = bert_mask[content_slice].sum().item()             # centers + neighbors
n_centers    = (bert_label[content_slice] != -100).sum().item()  # loss-bearing positions
rate_corr    = n_corrupted / max(n_content, 1)
rate_centers = n_centers   / max(n_content, 1)

print(f"Content length (between <sos> and <eos>):        {n_content}")
print(f"Positions corrupted in input (centers + nbrs):   {n_corrupted}  ({rate_corr:.1%})")
print(f"Positions with loss label    (centers only):     {n_centers}  ({rate_centers:.1%})")
print()
print("Under spaced-center masking, neighbors are deterministically [MASK]'d to")
print("close the 3-mer overlap leak, but loss is computed only at centers. With")
print("k=3 spacing >= 3 between centers, expect ~15% center rate and ~38% effective")
print("corruption rate.")
print()

# Corruption fate split, computed only at CENTER positions (neighbors are always [MASK]).
center_positions = (bert_label != -100).nonzero(as_tuple=True)[0]
n_total    = len(center_positions)
n_mask_tok = (bert_input[center_positions] == vocab.mask_index).sum().item()
n_kept     = (bert_input[center_positions] == bert_label[center_positions]).sum().item()
n_random   = n_total - n_mask_tok - n_kept

print("Corruption breakdown at CENTER positions:")
print(f"  [MASK]  : {n_mask_tok:>4}  ({n_mask_tok / max(n_total,1):.1%})    expected ~80%")
print(f"  random  : {n_random:>4}  ({n_random   / max(n_total,1):.1%})    expected ~10%")
print(f"  keep    : {n_kept:>4}  ({n_kept     / max(n_total,1):.1%})    expected ~10%")

Content length (between <sos> and <eos>):  510
Positions selected for MLM loss:            183  (35.9%)

Note: the dataset uses k-mer neighbor expansion, so the *effective* masked
rate after expansion is higher than the 15% Bernoulli rate. With k=3 each
masked center expands to itself + 2 neighbors, so expect ~30-40% effective.

Corruption breakdown over MLM-selected positions:
  [MASK]  :  170  (92.9%)    expected ~80%
  random  :    7  (3.8%)    expected ~10%
  keep    :    6  (3.3%)    expected ~10%


## Sanity invariants

Things that *must* be true if the masking is healthy. Any failure here is a bug.

In [ ]:
checks = []

# (1) Labels at center positions are real, in-vocab tokens.
center_mask = bert_label != -100
real_labels = bert_label[center_mask]
checks.append(("All non-(-100) labels are real, in-vocab tokens",
               ((real_labels >= 0) & (real_labels < len(vocab))).all().item()))

# (2) Center selection rate is reasonable (~15% Bernoulli over content).
content_len = eos_pos - 1
n_centers   = center_mask.sum().item()
center_rate = n_centers / max(content_len, 1)
checks.append((f"Center rate in [5%, 25%] (got {center_rate:.1%})",
               0.05 <= center_rate <= 0.25))

# (3) Centers are spaced at least k apart.
center_idx = center_mask.nonzero(as_tuple=True)[0].tolist()
gaps = [center_idx[i+1] - center_idx[i] for i in range(len(center_idx)-1)]
checks.append((f"Min center spacing >= {K} (got {min(gaps) if gaps else 'N/A'})",
               len(gaps) == 0 or min(gaps) >= K))

# (4) No special tokens were selected as centers.
special_token_ids = {vocab.pad_index, vocab.unk_index, vocab.eos_index, vocab.sos_index, vocab.mask_index}
checks.append(("No special tokens selected as centers",
               all(t not in special_token_ids for t in real_labels.tolist())))

# (5) Methylation is hidden at all corrupted positions (centers + neighbors).
checks.append(("Methylation = UNKNOWN at all corrupted positions",
               (methyl_seq[bert_mask] == STATE_UNKNOWN).all().item()))

# (6) Neighbor positions (in bert_mask, not in centers) all have input == [MASK].
neighbor_mask = bert_mask & ~center_mask
checks.append(("All neighbor positions corrupted to [MASK]",
               (bert_input[neighbor_mask] == vocab.mask_index).all().item()))

# (7) <sos>/<eos> sanity.
checks.append(("<sos> at position 0", bert_input[0].item() == vocab.sos_index))
checks.append(("Exactly one <eos> in the sequence",
               (bert_input == vocab.eos_index).sum().item() == 1))

for name, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

  PASS  All non-masked labels are -100
  FAIL  All masked labels are real tokens (>=0)
  PASS  No <pad>/<sos>/<eos> selected for MLM
  PASS  Methylation hidden at all masked positions
  PASS  <sos> at position 0
  PASS  Exactly one <eos> in the sequence


## Per-position view

Position-by-position table for the content region (between `<sos>` and `<eos>`). Markers in the rightmost column:

- `[MASK]` &mdash; replaced with the mask token (80% case)
- `RAND`  &mdash; replaced with a random k-mer (10% case)
- `KEEP`  &mdash; kept original but label is set, model still predicts here (10% case)
- blank  &mdash; not selected for the MLM objective

In [ ]:
def decode(idx):
    if idx < 0:
        return "---"
    return vocab.itos[idx] if idx < len(vocab) else f"<{idx}>"

def corruption_kind(input_id, label_id, was_masked):
    """
    Marker meanings:
      [MASK]·C : center, replaced with [MASK] (~80% of centers)
      RAND·C   : center, replaced with constrained random k-mer (~10%)
      KEEP·C   : center, kept original — model still predicts here (~10%)
      [MASK]·n : neighbor of a center, deterministically [MASK]'d
      (blank)  : not corrupted
    """
    if not was_masked:
        return ""
    is_center = (label_id != -100)
    if input_id == vocab.mask_index:
        return "[MASK]·C" if is_center else "[MASK]·n"
    if is_center and input_id == label_id:
        return "KEEP·C"
    if is_center:
        return "RAND·C"
    return "?"   # shouldn't happen: a non-center, non-[MASK] corrupted position

SHOW_PAD = False
end_view = eos_pos + 1 if not SHOW_PAD else len(bert_input)

header = f"{'pos':>4}  {'orig':>6}  {'input':>6}  {'label':>6}  {'methyl':>8}  marker"
print(header)
print("-" * len(header))

for pos in range(end_view):
    inp = bert_input[pos].item()
    lab = bert_label[pos].item()
    mst = methyl_seq[pos].item()
    msk = bert_mask[pos].item()
    # Recoverable original: at centers we have the label; at neighbor positions
    # the original was overwritten with [MASK] and isn't preserved in the dict
    # (we'd need to re-fetch from the memmap to see it). Elsewhere input == original.
    orig = lab if lab != -100 else inp
    print(f"{pos:>4}  {decode(orig):>6}  {decode(inp):>6}  "
          f"{(decode(lab) if lab != -100 else '---'):>6}  "
          f"{STATE_NAMES.get(mst, str(mst)):>8}  "
          f"{corruption_kind(inp, lab, msk)}")

 pos    orig   input   label    methyl  marker
----------------------------------------------
   0   <sos>   <sos>     ---   NON_CPG  
   1     AAA     AAA     ---   NON_CPG  
   2     AAT     AAT     ---   NON_CPG  
   3     ATA     ATA     ---   NON_CPG  
   4     TAC     TAC     ---   NON_CPG  
   5     ACT     ACT     ---   NON_CPG  
   6     CTT     CTT     ---   NON_CPG  
   7     ---  <mask>     ---   UNKNOWN  [MASK]
   8     TAA  <mask>     TAA   UNKNOWN  [MASK]
   9     ---  <mask>     ---   UNKNOWN  [MASK]
  10     ---  <mask>     ---   UNKNOWN  [MASK]
  11     CCT  <mask>     CCT   UNKNOWN  [MASK]
  12     ---  <mask>     ---   UNKNOWN  [MASK]
  13     TCT     TCT     ---   NON_CPG  
  14     CTG     CTG     ---   NON_CPG  
  15     TGA     TGA     ---   NON_CPG  
  16     GAC     GAC     ---   NON_CPG  
  17     ACT     ACT     ---   NON_CPG  
  18     CTC     CTC     ---   NON_CPG  
  19     TCC     TCC     ---   NON_CPG  
  20     ---  <mask>     ---   UNKNOWN  [MASK]
  2

## Aggregate over many samples

A single sample is noisy &mdash; a 15% Bernoulli over a 500-position content region has a stddev of ~8 positions. Average over a few hundred samples to see if the rates match the spec in expectation.

In [ ]:
N_SAMPLES = 300
rng = np.random.default_rng(0)
indices = rng.choice(len(dataset), size=min(N_SAMPLES, len(dataset)), replace=False)

totals = Counter()
for j in indices:
    s = dataset[int(j)]
    inp, lab, msk = s["bert_input"].long(), s["bert_label"].long(), s["bert_mask"].bool()

    eos_p = (inp == vocab.eos_index).nonzero(as_tuple=True)[0]
    eos_p = eos_p[0].item() if len(eos_p) > 0 else len(inp) - 1
    content = slice(1, eos_p)

    totals["content_positions"] += eos_p - 1
    totals["corrupted"]          += msk[content].sum().item()
    totals["centers"]            += (lab[content] != -100).sum().item()

    # 80/10/10 over centers only.
    center_positions = (lab != -100).nonzero(as_tuple=True)[0]
    totals["center_total"]       += len(center_positions)
    totals["center_mask_token"]  += (inp[center_positions] == vocab.mask_index).sum().item()
    totals["center_keep"]        += (inp[center_positions] == lab[center_positions]).sum().item()

totals["center_random"] = (
    totals["center_total"] - totals["center_mask_token"] - totals["center_keep"]
)

corr_rate   = totals["corrupted"] / max(totals["content_positions"], 1)
center_rate = totals["centers"]   / max(totals["content_positions"], 1)
ct          = max(totals["center_total"], 1)

print(f"Averaged over {len(indices)} samples:")
print(f"  Input-corruption rate (centers + nbrs): {corr_rate:.2%}    expected ~38%")
print(f"  Center selection rate (loss-bearing):   {center_rate:.2%}    expected ~15%")
print()
print("Corruption fate split at center positions:")
print(f"  [MASK] share: {totals['center_mask_token']/ct:.2%}    expected ~80%")
print(f"  random share: {totals['center_random']    /ct:.2%}    expected ~10%")
print(f"  keep   share: {totals['center_keep']      /ct:.2%}    expected ~10%")

Averaged over 300 samples:
  Effective MLM-selection rate (post k-mer expansion): 14.84%
  [MASK] share: 79.50%    expected ~80%
  random share: 10.20%    expected ~10%
  keep   share: 10.31%    expected ~10%
